# Supervised Learning Task 2: Multiple Linear Regression

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score

In [2]:
# Load the dataset
df = pd.read_csv("diabetes_dirty.csv")
df.head()

,AGE,SEX,BMI,BP,S1,S2,S3,S4,S5,S6,PROGRESSION
0,59,2,32.1,101.0,157,93.2,38.0,4.0,4.8598,87,151
1,48,1,21.6,87.0,183,103.2,70.0,3.0,3.8918,69,75
2,72,2,30.5,93.0,156,93.6,41.0,4.0,4.6728,85,141
3,24,1,25.3,84.0,198,131.4,40.0,5.0,4.8903,89,206
4,50,1,23.0,101.0,192,125.4,52.0,4.0,4.2905,80,135


## 1. Inspect the Data

Before cleaning or modelling, we check the shape of the dataset, the data
type of each column, and how many missing values are present in each one.

In [3]:
# Check shape, data types, and missing values
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())

(442, 11)
AGE              int64
SEX              int64
BMI            float64
BP             float64
S1               int64
S2             float64
S3             float64
S4             float64
S5             float64
S6               int64
PROGRESSION      int64
dtype: object
AGE            0
SEX            0
BMI            0
BP             0
S1             0
S2             0
S3             0
S4             0
S5             0
S6             0
PROGRESSION    0
dtype: int64


In [4]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check basic statistics for outliers
df.describe()

Duplicate rows: 0


,AGE,SEX,BMI,BP,S1,S2,S3,S4,S5,S6,PROGRESSION
count,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000
mean,48.518100,1.468326,26.375792,94.647014,189.140271,115.439140,49.788462,4.070249,4.641411,91.260181,152.133484
std,13.109028,0.499561,4.418122,13.831283,34.608052,30.413081,12.934202,1.290450,0.522391,11.496335,77.093005
min,19.000000,1.000000,18.000000,62.000000,97.000000,41.600000,22.000000,2.000000,3.258100,58.000000,25.000000
25%,38.250000,1.000000,23.200000,84.000000,164.250000,96.050000,40.250000,3.000000,4.276700,83.250000,87.000000
50%,50.000000,1.000000,25.700000,93.000000,186.000000,113.000000,48.000000,4.000000,4.620050,91.000000,140.500000
75%,59.000000,2.000000,29.275000,105.000000,209.750000,134.500000,57.750000,5.000000,4.997200,98.000000,211.500000
max,79.000000,2.000000,42.200000,133.000000,301.000000,242.400000,99.000000,9.090000,6.107000,124.000000,346.000000


## 2. Define Inputs and Target Variable

PROGRESSION is what we want to predict, so it becomes our target (y).
All other columns are independent variables and become our inputs (X).

In [5]:
# Separate independent variables (X) and target variable (y)
X = df.drop(columns=["PROGRESSION"])
y = df["PROGRESSION"]

print(X.shape)
print(y.shape)

(442, 10)
(442,)


## 3. Split into Training and Test Sets

We keep 80% of the data for training and hold back 20% for testing.
The model never sees the test set during training, so it gives us an honest
measure of how well the model generalises to new data.

In [6]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (353, 10)
Test set: (89, 10)


## 4. Feature Scaling

The features are on different scales, so we normalise them using MinMaxScaler.
We fit the scaler on the training set only, then apply it to both sets.
This prevents any information from the test set leaking into the model.

In [7]:
# Fit scaler on training data only
scaler = MinMaxScaler()
scaler.fit(X_train)

# Apply to both training and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

Scaling complete.


## 5. Fit the Multiple Linear Regression Model

We train the model on the scaled training data. Once fitted, we print the
intercept and coefficients. Each coefficient tells us how much PROGRESSION
changes for a one-unit increase in that feature, holding all others constant.

In [8]:
# Fit the model on scaled training data
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Print intercept and coefficients
print("Intercept:", round(model.intercept_, 4))
print("\nCoefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {round(coef, 4)}")

Intercept: -25.1332

Coefficients:
  AGE: 8.2613
  SEX: -23.0645
  BMI: 135.6356
  BP: 84.9936
  S1: -244.8018
  S2: 162.8793
  S3: 46.3273
  S4: 72.0311
  S5: 191.1867
  S6: 13.3055


## 6. Predictions and Model Evaluation

We generate predictions on the test set and compute R squared. R squared
measures how much of the variation in PROGRESSION the model explains.
A score of 1.0 is a perfect fit. A score of 0 means the model does no
better than simply predicting the average every time.

In [9]:
# Generate predictions on the test set
y_pred = model.predict(X_test_scaled)

# Compute R squared
r2 = r2_score(y_test, y_pred)
print(f"R squared: {round(r2, 4)}")

R squared: 0.4526


## 7. Model Interpretation

The model returned an R squared score of 0.4526 on the test set. This means
the model accounts for roughly 45% of the variation in diabetes progression.
The remaining 55% is likely driven by factors not captured in this dataset,
or by non-linear relationships that a simple linear model cannot detect.
BMI, S5 (blood serum measurement), and S2 (LDL cholesterol) had the largest
positive effect on progression. S1 (total cholesterol) had the largest
negative coefficient, though this is partly because S1 and S2 are correlated
and their effects partially cancel out.